# 18 — Branch, WMH-volume and LOCO panels

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "analysis" / "lib").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from scipy.stats import t as t_dist
from analysis.lib.stats import nadeau_bengio_test, bh_fdr
from analysis.lib.data import restricted_pool
from analysis.lib.plotting import (PALETTE, VARIABLE_LABELS, DISPLAY_NAMES,
                                   COHORT_LABELS, SIG_ALPHA,
                                   compute_nb_sig_table, get_sig_flags,
                                   plot_sig_markers, sig_legend_entries,
                                   apply_manuscript_style)

BR          = PROJECT_ROOT / "outputs" / "predictive_modeling_branches"
BR_FIG      = BR / "figures"
WMH_VT      = PROJECT_ROOT / "outputs" / "predictive_modeling_wmh" / "volume_vs_topology"
STRATA_DIR  = PROJECT_ROOT / "outputs" / "predictive_modeling_volume_strata"
EXP         = PROJECT_ROOT / "outputs" / "subject_level_analyses"
EXP_FIG     = EXP / "figures"
LOCO        = PROJECT_ROOT / "outputs" / "predictive_modeling_core" / "loco"
for d in (BR_FIG, WMH_VT / "figures", STRATA_DIR / "figures", EXP_FIG):
    d.mkdir(parents=True, exist_ok=True)

CLIN_PATH  = PROJECT_ROOT / "data" / "tabular" / "clinical_data.csv"
LACUNA_DIR = PROJECT_ROOT / "data" / "lacuna" / "METAVCI_PSCI_LACUNA" / "derivatives"
assert CLIN_PATH.exists(), f"missing {CLIN_PATH} — see data/README.md"

In [ ]:
# Branch ΔAUPRC bars (Fig 5a main + supplement)
BRANCHES = ["acute_vs_wmh", "acute_vs_oldinfarct", "acute_vs_lacune", "acute_vs_chronic"]
BRANCH_LABELS = {
    "acute_vs_wmh":        "+ WMH",
    "acute_vs_oldinfarct": "+ old infarct",
    "acute_vs_lacune":     "+ lacune",
    "acute_vs_chronic":    "+ all pre-existing lesion types joined",
}
BRANCH_PALETTE = {
    "acute_vs_wmh":        "#1b9e77",
    "acute_vs_oldinfarct": "#d95f02",
    "acute_vs_lacune":     "#7570b3",
    "acute_vs_chronic":    "#e7298a",
}

stacker = {b: pd.read_csv(BR / b / "stacker_scores.csv")
           for b in BRANCHES if (BR / b / "stacker_scores.csv").exists()}
print(f"loaded {len(stacker)} branch stacker CSVs")

rows = []
for b, df in stacker.items():
    a = df[df["Arm"].str.startswith("a_")].set_index(["Target", "Fold"])["AUPRC"]
    bb = df[df["Arm"].str.startswith("b_")].set_index(["Target", "Fold"])["AUPRC"]
    for tgt in sorted(df["Target"].unique()):
        try:
            a_t = a.xs(tgt, level="Target")
            b_t = bb.xs(tgt, level="Target")
        except KeyError:
            continue
        common = a_t.index.intersection(b_t.index)
        if len(common) < 2:
            continue
        d = (b_t.loc[common] - a_t.loc[common]).dropna()
        rows.append({"Branch": b, "Target": tgt,
                     "mean_delta": float(d.mean()),
                     "se_delta": float(d.std(ddof=1) / np.sqrt(len(d))) if len(d) > 1 else np.nan,
                     "n_paired": int(len(d))})
branch_stacker_stats = pd.DataFrame(rows)

clinical_df = pd.read_csv(CLIN_PATH, index_col=0)
BRANCH_TO_LABEL = {"acute_vs_wmh": "WMH", "acute_vs_oldinfarct": "oldinfarct",
                   "acute_vs_lacune": "lacune"}
POOL_SIZES = {}
for b in BRANCHES:
    if b == "acute_vs_chronic":
        POOL_SIZES[b] = len(restricted_pool(LACUNA_DIR, clinical_df, ["acuteinfarct"],
                                            any_of=["WMH", "oldinfarct", "lacune"]))
    else:
        POOL_SIZES[b] = len(restricted_pool(LACUNA_DIR, clinical_df,
                                            ["acuteinfarct", BRANCH_TO_LABEL[b]]))

sns.set_theme(style="whitegrid", font_scale=1.3,
              rc={"font.weight": "bold", "axes.labelweight": "bold",
                  "axes.titleweight": "bold"})


def plot_branch_deltas(branches_to_plot, save_path):
    branches_present = [b for b in branches_to_plot
                        if b in branch_stacker_stats["Branch"].unique()]
    targets_present = [t for t in VARIABLE_LABELS
                       if t in branch_stacker_stats["Target"].unique()]
    n_targets = len(targets_present)
    n_branches = len(branches_present)
    if n_targets == 0 or n_branches == 0:
        print(f"nothing to plot for {branches_to_plot}")
        return None

    group_w = 0.7 if n_branches <= 2 else 0.8
    band_w = group_w / n_branches
    offsets = np.linspace(-group_w / 2 + band_w / 2, group_w / 2 - band_w / 2, n_branches)
    fig, ax = plt.subplots(figsize=(max(16, 2.9 * n_targets), 8.5))

    for x_idx, tgt in enumerate(targets_present):
        for g_idx, branch in enumerate(branches_present):
            cx = x_idx + offsets[g_idx]
            row = branch_stacker_stats[
                (branch_stacker_stats["Branch"] == branch)
                & (branch_stacker_stats["Target"] == tgt)]
            if row.empty:
                continue
            mean = float(row["mean_delta"].iloc[0])
            se = float(row["se_delta"].iloc[0]) if pd.notna(row["se_delta"].iloc[0]) else 0.0
            hatch = "////" if mean < 0 else None
            ax.bar(cx, mean, width=band_w * 0.95, color=BRANCH_PALETTE[branch],
                   edgecolor="black", linewidth=0.6, hatch=hatch,
                   label=(f"{BRANCH_LABELS[branch]}  (n={POOL_SIZES[branch]})"
                          if x_idx == 0 else None))
            if se > 0:
                ax.errorbar(cx, mean, yerr=se, color="black", linewidth=0.8,
                            capsize=3, zorder=4)
            text_off = 0.004 if mean >= 0 else -0.004
            text_y = mean + (se if mean >= 0 else -se) + text_off
            ax.text(cx, text_y, f"{mean:+.3f}", ha="center",
                    va="bottom" if mean >= 0 else "top",
                    fontsize=15, fontweight="bold", color="#222", rotation=90)

    ax.axhline(0, color="black", linewidth=0.7)
    for x in range(1, n_targets):
        ax.axvline(x - 0.5, color="gray", linewidth=0.3, alpha=0.4)

    sub_means = branch_stacker_stats[
        branch_stacker_stats["Branch"].isin(branches_present)
        & branch_stacker_stats["Target"].isin(targets_present)]
    if len(sub_means):
        y_lo = float((sub_means["mean_delta"] - sub_means["se_delta"].fillna(0)).min())
        y_hi = float((sub_means["mean_delta"] + sub_means["se_delta"].fillna(0)).max())
        pad = 0.035
        ax.set_ylim(min(y_lo, 0) - pad, max(y_hi, 0) + pad)

    ax.set_xticks(range(n_targets))
    ax.set_xticklabels([VARIABLE_LABELS[t] for t in targets_present],
                       fontsize=16, rotation=0)
    ax.tick_params(axis="y", labelsize=15)
    ax.set_ylabel("Performance difference\n(ΔAUPRC, acute and pre-existing − acute-only)",
                  fontsize=16, fontweight="bold")
    ax.set_xlabel("")

    handles, labels = ax.get_legend_handles_labels()
    handles.append(Patch(facecolor="white", edgecolor="black", hatch="////",
                         label="Δ < 0 (pre-existing features hurt)"))
    labels.append("Δ < 0 (pre-existing features hurt)")
    ax.legend(handles, labels, bbox_to_anchor=(0.5, -0.16), loc="upper center",
              ncol=2, frameon=True, fontsize=15)
    plt.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    fig.savefig(save_path.with_suffix(".svg"), bbox_inches="tight")
    print(f"saved {save_path}")
    return fig


plot_branch_deltas(["acute_vs_wmh", "acute_vs_chronic"],
                   BR_FIG / "branch_deltas_main_wmh_chronic.png")
plot_branch_deltas(BRANCHES,
                   BR_FIG / "branch_deltas_supplement_all_branches.png")

In [ ]:
# WMH volume vs topology violin (Fig 5d)
apply_manuscript_style()

LOCAL_SIG_MARKERS = [("Majority", "^"), ("Permuted", "s"), ("Volume", "p")]


def _local_sig_flags(nb_df, target, focal):
    out = {ref: False for ref, _ in LOCAL_SIG_MARKERS}
    sub = nb_df[(nb_df["Target"] == target) & (nb_df["Focal"] == focal)]
    for ref, _ in LOCAL_SIG_MARKERS:
        row = sub[sub["Reference"] == ref]
        if len(row) == 0:
            continue
        p = row["p_adj_BH"].iloc[0] if "p_adj_BH" in row.columns else row["p"].iloc[0]
        d = row["mean_diff"].iloc[0]
        if pd.notna(p) and p < SIG_ALPHA and pd.notna(d) and d > 0:
            out[ref] = True
    return out


def _local_plot_markers(ax, cx, flags, base_y, row_step, marker_size, color):
    for i, (ref, mk) in enumerate(LOCAL_SIG_MARKERS):
        if flags[ref]:
            ax.scatter(cx, base_y + i * row_step, marker=mk, s=marker_size,
                       color=color, edgecolor="black", linewidth=0.8,
                       zorder=6, clip_on=False)


def _local_sig_legend():
    label_map = {"Majority": "Majority", "Permuted": "Permuted",
                 "Volume": "WMH volume"}
    return [Line2D([0], [0], marker=mk, color="w", markerfacecolor="black",
                   markeredgecolor="black", markersize=9, linewidth=0,
                   label=f"> {label_map[ref]} ($p_{{FDR}}<{SIG_ALPHA}$, NB)")
            for ref, mk in LOCAL_SIG_MARKERS]


VT_GROUPS = ["Majority", "Permuted", "Volume", "Topology"]
VT_PAL = {"Majority": PALETTE["Majority"], "Permuted": PALETTE["Permuted"],
          "Volume": "#F0D38A", "Topology": "#7FC8B9"}
VT_DISPLAY = {"Majority": "Majority", "Permuted": "Permuted",
              "Volume": "WMH volume",
              "Topology": "WMH topography and connectivity"}

all_scores = pd.read_csv(WMH_VT / "all_scores.csv")
stk_scores = pd.read_csv(WMH_VT / "stacking_scores.csv")

base = all_scores[all_scores["Modality"] == "base"].copy()
base["Group"] = base["Model"]
vol = all_scores[all_scores["Modality"] == "wmh_volume"].copy()
vol["Group"] = "Volume"
stk = stk_scores.copy()
stk["Group"] = "Topology"
pooled = pd.concat([
    base[["Target", "Group", "Fold", "AUPRC", "AUPRC_train"]],
    vol[["Target", "Group", "Fold", "AUPRC", "AUPRC_train"]],
    stk[["Target", "Group", "Fold", "AUPRC", "AUPRC_train"]],
], ignore_index=True)

nb = compute_nb_sig_table(pooled, focal_groups=["Topology"],
                          reference_groups=["Majority", "Permuted", "Volume"],
                          metric="AUPRC")
nb.to_csv(WMH_VT / "nb_significance.csv", index=False)
print(nb.round(4).to_string(index=False))

vol_df = pd.read_csv(PROJECT_ROOT / "outputs" / "predictive_modeling_wmh" / "wmh_volumes.csv")
wmh_pool = clinical_df.index.intersection(vol_df["subject_id"])
n_per_target_wmh = {t: int(clinical_df.loc[wmh_pool, t].notna().sum())
                    for t in VARIABLE_LABELS if t in clinical_df.columns}

targets_list = [t for t in VARIABLE_LABELS if t in pooled["Target"].unique()]
n_targets = len(targets_list)
n_groups = len(VT_GROUPS)

sns.set_theme(style="whitegrid", font_scale=1.5,
              rc={"font.weight": "bold", "axes.labelweight": "bold",
                  "axes.titleweight": "bold"})
fig, ax = plt.subplots(figsize=(22, 8.5))
fig.subplots_adjust(left=0.055, right=0.985, top=0.95, bottom=0.26)
sns.violinplot(data=pooled, x="Target", y="AUPRC", hue="Group",
               hue_order=VT_GROUPS, palette=VT_PAL,
               linewidth=0, saturation=0.7, cut=0, inner=None, width=0.85, ax=ax)

group_w = 0.85
band_w = group_w / n_groups
offsets = np.linspace(-group_w / 2 + band_w / 2, group_w / 2 - band_w / 2, n_groups)
train_means = pooled.groupby(["Target", "Group"])["AUPRC_train"].mean()

ax.set_ylim(0, 1.18)
sig_base_y = 1.02
sig_step = 0.045
rng = np.random.RandomState(0)
for x_idx, tname in enumerate(targets_list):
    for g_idx, g in enumerate(VT_GROUPS):
        cx = x_idx + offsets[g_idx]
        vals = pooled.loc[(pooled["Target"] == tname) & (pooled["Group"] == g),
                          "AUPRC"].values
        if len(vals):
            jx = rng.normal(loc=cx, scale=0.015, size=len(vals))
            ax.scatter(jx, vals, s=8, color="gray", alpha=0.35, zorder=2)
            ax.scatter(cx, np.nanmean(vals), color="#222", marker="D", s=55, zorder=3)
            tr = train_means.get((tname, g), np.nan)
            if np.isfinite(tr):
                ax.scatter(cx, tr, s=55, zorder=3, facecolors="none",
                           edgecolors="#222", linewidths=1.4, marker="o")
        if g == "Topology" and len(nb):
            flags = _local_sig_flags(nb, tname, "Topology")
            _local_plot_markers(ax, cx, flags, sig_base_y, sig_step,
                                marker_size=70, color=VT_PAL[g])
        if len(vals):
            ax.text(cx, sig_base_y - 0.035, f"{np.nanmean(vals):.2f}",
                    ha="center", va="top", fontsize=13, fontweight="bold",
                    color="#222", zorder=5)

ax.set_xticks(range(n_targets))
ax.set_xticklabels([f"{VARIABLE_LABELS[t]}\n(n={n_per_target_wmh.get(t, '?')})"
                    for t in targets_list], fontsize=16, rotation=0)
ax.tick_params(axis="y", labelsize=15)
ax.set_ylabel("AUPRC (per fold)", fontsize=17, fontweight="bold")
ax.set_xlabel("")

extra_handles = [
    Line2D([0], [0], marker="D", color="w", markerfacecolor="#222",
           markersize=9, linewidth=0, label="Mean test AUPRC"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="none",
           markeredgecolor="#222", markersize=9, markeredgewidth=1.4,
           linewidth=0, label="Mean train AUPRC"),
] + _local_sig_legend()
mh, ml = ax.get_legend_handles_labels()
ax.legend(mh + extra_handles,
          [VT_DISPLAY.get(l, l) for l in ml] + [h.get_label() for h in extra_handles],
          bbox_to_anchor=(0.5, -0.20), loc="upper center",
          frameon=True, ncol=5, fontsize=13)
plt.tight_layout()
out_png = WMH_VT / "figures" / "wmh_volume_vs_topology_violin.png"
fig.savefig(out_png, dpi=200, bbox_inches="tight")
fig.savefig(out_png.with_suffix(".svg"), bbox_inches="tight")
fig.savefig(out_png.with_suffix(".pdf"), bbox_inches="tight")
print(f"saved {out_png}")

In [ ]:
# Volume-strata NB table + ΔAUPRC strip (Fig 2e)
STRATUM_ORDER = ["small", "medium", "large"]
CORE_GROUPS = ["Majority", "Permuted", "Baseline", "Stacking"]

strata_all = pd.read_csv(STRATA_DIR / "all_scores.csv")
strata_stk = pd.read_csv(STRATA_DIR / "stacking_scores.csv")

nb_rows = []
for stratum in STRATUM_ORDER:
    df_a = strata_all[strata_all["Stratum"] == stratum]
    df_s = strata_stk[strata_stk["Stratum"] == stratum]
    if df_a.empty or df_s.empty:
        continue
    base = df_a[df_a["Modality"] == "base"].copy()
    base["Group"] = base["Model"]
    stk_s = df_s.copy()
    stk_s["Group"] = "Stacking"
    pl = pd.concat([base[["Target", "Group", "Fold", "AUPRC", "AUPRC_train"]],
                    stk_s[["Target", "Group", "Fold", "AUPRC", "AUPRC_train"]]],
                   ignore_index=True)
    nb_s = compute_nb_sig_table(pl, focal_groups=["Stacking"],
                                reference_groups=["Majority", "Permuted", "Baseline"],
                                metric="AUPRC")
    nb_s["Stratum"] = stratum
    nb_rows.append(nb_s)
nb_all = pd.concat(nb_rows, ignore_index=True)
# FDR family for integrated > baseline: the strata within each cognitive target
is_b = nb_all["Reference"] == "Baseline"
nb_all.loc[is_b, "p_adj_BH"] = (nb_all[is_b].groupby("Target")["p"]
                                .transform(lambda s: bh_fdr(s.values)))
nb_all.to_csv(STRATA_DIR / "nb_significance_per_stratum.csv", index=False)
print(f"saved {STRATA_DIR / 'nb_significance_per_stratum.csv'}")

# strip plot
sns.set_theme(style="whitegrid", font_scale=1.3,
              rc={"font.weight": "bold", "axes.labelweight": "bold",
                  "axes.titleweight": "bold"})

# Stratum Ns from the actual stratified pools
acute_pool = restricted_pool(LACUNA_DIR, clinical_df, ["acuteinfarct"])
# count the analysed sample (patients with an any-domain PSCI label)
acute_pool = [p for p in acute_pool
              if pd.notna(clinical_df.loc[p, "cognition-PSCI_present"])]
v = pd.to_numeric(clinical_df.loc[acute_pool, "stroke-total_infarct_volume_ml"],
                  errors="coerce")
strat_n = {"small": int((v < 5).sum()),
           "medium": int(((v >= 5) & (v < 30)).sum()),
           "large": int((v >= 30).sum())}
STRATUM_LABEL = {
    "small":  f"Small (< 5 mL,  N = {strat_n['small']})",
    "medium": f"Medium (5–30 mL,  N = {strat_n['medium']})",
    "large":  f"Large (≥ 30 mL,  N = {strat_n['large']})",
}
STRATUM_COLOR = {"small": "#7FB069", "medium": "#E6AA68", "large": "#C75A4A"}

nb_lookup = {(r["Stratum"], r["Target"]): r["p_adj_BH"]
             for _, r in nb_all[nb_all["Reference"] == "Baseline"].iterrows()}

rows = []
for stratum in STRATUM_ORDER:
    bl = strata_all[(strata_all["Stratum"] == stratum)
                    & (strata_all["Model"] == "Baseline")
                    & (strata_all["Modality"] == "base")]
    stk_s = strata_stk[strata_stk["Stratum"] == stratum]
    for target in bl["Target"].unique():
        b = bl[bl["Target"] == target].set_index("Fold")["AUPRC"]
        s = stk_s[stk_s["Target"] == target].set_index("Fold")["AUPRC"]
        common = b.index.intersection(s.index)
        d = (s.loc[common] - b.loc[common]).dropna()
        if len(d) < 2:
            continue
        mean = float(d.mean())
        # 95% CI with the Nadeau-Bengio corrected variance (as in the test)
        se = float(np.sqrt((1 / len(d) + 0.1 / 0.9) * d.var(ddof=1)))
        ci = t_dist.ppf(0.975, len(d) - 1) * se
        rows.append({"stratum": stratum, "target": target, "delta": mean,
                     "ci_lo": mean - ci, "ci_hi": mean + ci, "n_folds": len(d)})
df = pd.DataFrame(rows)

targets_list = [t for t in VARIABLE_LABELS if t in df["target"].unique()]
n_targets = len(targets_list)
fig, ax = plt.subplots(figsize=(20, 7))
fig.subplots_adjust(left=0.10, right=0.95, top=0.92, bottom=0.22)
x_pos = np.arange(n_targets)
offsets = {"small": -0.22, "medium": 0.0, "large": +0.22}

for stratum in STRATUM_ORDER:
    sub = df[df["stratum"] == stratum].set_index("target")
    xs = x_pos + offsets[stratum]
    ys, ci_los, ci_his = [], [], []
    for t in targets_list:
        if t in sub.index:
            ys.append(sub.loc[t, "delta"])
            ci_los.append(sub.loc[t, "ci_lo"])
            ci_his.append(sub.loc[t, "ci_hi"])
        else:
            ys.append(np.nan); ci_los.append(np.nan); ci_his.append(np.nan)
    ys = np.asarray(ys); ci_los = np.asarray(ci_los); ci_his = np.asarray(ci_his)
    err_lo = ys - ci_los
    err_hi = ci_his - ys
    ax.errorbar(xs, ys, yerr=[err_lo, err_hi], fmt="o", ms=12,
                mfc=STRATUM_COLOR[stratum], mec="black", mew=1.0,
                ecolor=STRATUM_COLOR[stratum], elinewidth=2.0,
                capsize=4, zorder=3, label=STRATUM_LABEL[stratum])
    for xi, t, yi in zip(xs, targets_list, ys):
        if np.isnan(yi):
            continue
        p = nb_lookup.get((stratum, t), np.nan)
        if pd.notna(p) and p < 0.05:
            ax.text(xi, yi + (err_hi[targets_list.index(t)] + 0.005), "*",
                    ha="center", va="bottom", fontsize=14, fontweight="bold",
                    color="#222")

ax.axhline(0, color="black", lw=0.6, alpha=0.6)
ax.set_xticks(x_pos)
ax.set_xticklabels([VARIABLE_LABELS[t] for t in targets_list], fontsize=13, rotation=0)
ax.set_ylabel("Performance difference \n(ΔAUPRC, integrated − baseline)")
ax.grid(True, axis="y", alpha=0.25, ls=":")
ymin = min(df["ci_lo"].min(), 0) - 0.02
ymax = df["ci_hi"].max() + 0.04
ax.set_ylim(ymin, ymax)
star_proxy = Line2D([0], [0], marker="", linestyle="",
                    label="$*$  > Baseline  ($p_{FDR}<0.05$, NB t-test)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.20), ncol=4, fontsize=12,
          frameon=True, handlelength=1.5, handletextpad=0.6, columnspacing=1.5,
          handles=[*ax.get_legend_handles_labels()[0], star_proxy])
plt.tight_layout()
out_png = STRATA_DIR / "figures" / "volume_strata_delta_strip.png"
fig.savefig(out_png, dpi=200, bbox_inches="tight")
fig.savefig(out_png.with_suffix(".svg"), bbox_inches="tight")
fig.savefig(out_png.with_suffix(".pdf"), bbox_inches="tight")
print(f"saved {out_png}")

In [ ]:
# LOCO PSCI per-cohort prevalence-normalised stats (cached on CSV)
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy.stats import wilcoxon, false_discovery_control

rng_np = np.random.default_rng(0)
df_pred = pd.read_csv(EXP / "loco_psci_per_subject_predictions.csv")
print(f"{len(df_pred)} per-subject rows across {df_pred.cohort.nunique()} cohorts")

CACHE = EXP / "loco_prevalence_normalised.csv"


def auprc_safe(y, s):
    return average_precision_score(y, s) if len(np.unique(y)) == 2 else np.nan


def auroc_safe(y, s):
    return roc_auc_score(y, s) if len(np.unique(y)) == 2 else np.nan


if CACHE.exists():
    print(f"[skip] {CACHE} exists")
    res = pd.read_csv(CACHE)
else:
    cohort_rows = []
    for cohort, sub in df_pred.groupby("cohort"):
        y = sub["y_true"].values
        s = sub["meta_decision"].values
        n = len(y)
        prev = y.mean()
        auprc = auprc_safe(y, s)
        auroc = auroc_safe(y, s)
        norm_auprc = (auprc - prev) / (1 - prev) if prev < 1 else np.nan

        n_boot = 1000
        boot = {"auprc": [], "auroc": [], "d_auprc": [], "norm_auprc": []}
        for _ in range(n_boot):
            idx = rng_np.integers(0, n, size=n)
            yb, sb = y[idx], s[idx]
            prevb = yb.mean()
            ap = auprc_safe(yb, sb)
            ar = auroc_safe(yb, sb)
            boot["auprc"].append(ap)
            boot["auroc"].append(ar)
            boot["d_auprc"].append(ap - prevb if not np.isnan(ap) else np.nan)
            boot["norm_auprc"].append(
                (ap - prevb) / (1 - prevb) if (not np.isnan(ap)) and prevb < 1 else np.nan)
        cis = {k: np.nanpercentile(v, [2.5, 97.5]) for k, v in boot.items()}

        n_perm = 10000
        perm_auprc = np.empty(n_perm)
        perm_auroc = np.empty(n_perm)
        for i in range(n_perm):
            yp = rng_np.permutation(y)
            perm_auprc[i] = auprc_safe(yp, s)
            perm_auroc[i] = auroc_safe(yp, s)
        cohort_rows.append({
            "cohort": cohort, "n": int(n), "prev": float(prev),
            "AUPRC": float(auprc),
            "AUPRC_lo": float(cis["auprc"][0]), "AUPRC_hi": float(cis["auprc"][1]),
            "AUROC": float(auroc),
            "AUROC_lo": float(cis["auroc"][0]), "AUROC_hi": float(cis["auroc"][1]),
            "delta_AUPRC": float(auprc - prev),
            "delta_AUPRC_lo": float(cis["d_auprc"][0]),
            "delta_AUPRC_hi": float(cis["d_auprc"][1]),
            "norm_AUPRC": float(norm_auprc),
            "norm_AUPRC_lo": float(cis["norm_auprc"][0]),
            "norm_AUPRC_hi": float(cis["norm_auprc"][1]),
            "p_perm_AUPRC": float((perm_auprc >= auprc).mean()),
            "p_perm_AUROC": float((perm_auroc >= auroc).mean())})
    res = pd.DataFrame(cohort_rows).sort_values("prev").reset_index(drop=True)
    res["p_perm_AUPRC_FDR"] = false_discovery_control(res["p_perm_AUPRC"].values,
                                                      method="bh")
    res["p_perm_AUROC_FDR"] = false_discovery_control(res["p_perm_AUROC"].values,
                                                      method="bh")
    res.to_csv(CACHE, index=False)
    print(f"saved {CACHE}")

w_auprc = wilcoxon(res["AUPRC"], res["prev"], alternative="greater", zero_method="wilcox")
w_auroc = wilcoxon(res["AUROC"] - 0.5, alternative="greater", zero_method="wilcox")
print(f"paired Wilcoxon AUPRC > prev: W={w_auprc.statistic:.1f} p={w_auprc.pvalue:.4f} "
      f"({int((res['AUPRC'] > res['prev']).sum())}/{len(res)} above)")
print(f"Wilcoxon AUROC > 0.5: W={w_auroc.statistic:.1f} p={w_auroc.pvalue:.4f} "
      f"({int((res['AUROC'] > 0.5).sum())}/{len(res)} above)")

In [ ]:
# LOCO imaging transfers (Fig S12b): stacker vs baseline normalised AUPRC
apply_manuscript_style()
from adjustText import adjust_text

res = pd.read_csv(CACHE)
res["cohort_display"] = res["cohort"].map(COHORT_LABELS).fillna(res["cohort"])

# LOCO fold index -> held-out cohort, per target, reproducing notebook 02:
# cohorts in first-appearance order on the acute pool, keeping only cohorts
# with >= 1 labelled patient for the target (folds are re-enumerated per target)
loco_all = pd.read_csv(LOCO / "all_scores.csv")
clin_pool = clinical_df.loc[restricted_pool(LACUNA_DIR, clinical_df, ["acuteinfarct"])]
COHORTS_LOCO = clin_pool["demographics-cohort"].dropna().unique().tolist()


def loco_fold_cohorts(target):
    lab = clin_pool[target].notna()
    kept = [c for c in COHORTS_LOCO if (lab & (clin_pool["demographics-cohort"] == c)).any()]
    f2c = dict(enumerate(kept))
    # check: Majority-model AUPRC of each fold equals the prevalence of the held-out
    # cohort (test) and of the remaining labelled cohorts (train); single-class
    # held-out cohorts have NaN test AUPRC, so only the train side is checked there
    maj = (loco_all[(loco_all["Target"] == target) & (loco_all["Model"] == "Majority")]
           .groupby("Fold")[["AUPRC", "AUPRC_train"]].mean())
    assert set(maj.index) == set(f2c), f"{target}: fold count mismatch"
    for f, c in f2c.items():
        in_c = clin_pool["demographics-cohort"] == c
        y_te, y_tr = clin_pool.loc[lab & in_c, target], clin_pool.loc[lab & ~in_c, target]
        ok_tr = abs(maj.loc[f, "AUPRC_train"] - y_tr.mean()) < 1e-9
        ok_te = (y_te.nunique() < 2 if pd.isna(maj.loc[f, "AUPRC"])
                 else abs(maj.loc[f, "AUPRC"] - y_te.mean()) < 1e-9)
        assert ok_tr and ok_te, f"{target}: fold {f} is not cohort {c}"
    return f2c


cohort_to_fold = {c: f for f, c in loco_fold_cohorts("cognition-PSCI_present").items()}

base_fold = (loco_all[(loco_all["Target"] == "cognition-PSCI_present")
                      & (loco_all["Model"] == "Baseline")]
             .groupby("Fold")["AUPRC"].mean().to_dict())
stacker_loco = pd.read_csv(LOCO / "stacking_scores.csv")
fold_auprc = (stacker_loco[stacker_loco["Target"] == "cognition-PSCI_present"]
              .groupby("Fold")["AUPRC"].mean().to_dict())

df = res.sort_values("prev").reset_index(drop=True)
df["baseline_AUPRC"] = df["cohort"].map(lambda c: base_fold[cohort_to_fold[c]])
df["baseline_norm_AUPRC"] = (df["baseline_AUPRC"] - df["prev"]) / (1 - df["prev"])
df["canon_full_AUPRC"] = df["cohort"].map(lambda c: fold_auprc[cohort_to_fold[c]])
df["norm_AUPRC"] = (df["canon_full_AUPRC"] - df["prev"]) / (1 - df["prev"])
df["delta_norm"] = df["norm_AUPRC"] - df["baseline_norm_AUPRC"]
print(df[["cohort_display", "n", "prev", "baseline_norm_AUPRC",
          "norm_AUPRC", "delta_norm"]].round(3).to_string(index=False))
w, p = wilcoxon(df["norm_AUPRC"], df["baseline_norm_AUPRC"], alternative="greater")
print(f"Wilcoxon stacker > baseline (norm_AUPRC): W={w:.1f}, p={p:.4f}, "
      f"{(df['delta_norm'] > 0).sum()}/{len(df)} positive")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.subplots_adjust(left=0.06, right=0.985, top=0.86, bottom=0.20, wspace=0.28)

ax = axes[0]
lo = min(df["baseline_norm_AUPRC"].min(), df["norm_AUPRC"].min()) - 0.04
hi = max(df["baseline_norm_AUPRC"].max(), df["norm_AUPRC"].max()) + 0.05
ax.plot([lo, hi], [lo, hi], color="#888", lw=0.8, ls=":", zorder=1,
        label="y = x  (no imaging lift)")
ax.scatter(df["baseline_norm_AUPRC"], df["norm_AUPRC"], s=80,
           c="#1f5fa8", edgecolor="black", linewidth=0.6, zorder=3)
texts = [ax.text(r["baseline_norm_AUPRC"], r["norm_AUPRC"], r["cohort_display"],
                 fontsize=9, fontweight="bold", color="#222", zorder=5)
         for _, r in df.iterrows()]
adjust_text(texts, ax=ax,
            arrowprops=dict(arrowstyle="-", color="#555", lw=0.5),
            expand=(1.5, 1.8), force_text=(0.5, 0.8))
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_xlabel("Predictive performance of baseline model\n(normalised AUPRC)",
              fontsize=12)
ax.set_ylabel("Predictive performance of integrated model\n(normalised AUPRC)",
              fontsize=12)
ax.legend(loc="lower right", frameon=False, fontsize=10)
ax.grid(True, alpha=0.2, ls=":")

ax = axes[1]
xs = np.arange(len(df))
colors = ["#aa3333" if d > 0 else "#3333aa" for d in df["delta_norm"]]
ax.bar(xs, df["delta_norm"], color=colors, edgecolor="black",
       linewidth=0.5, alpha=0.85)
ax.axhline(0, color="black", lw=0.6)
ax.set_xticks(xs)
ax.set_xticklabels([r.cohort_display for _, r in df.iterrows()],
                   rotation=35, ha="right", fontsize=10)
ax.set_ylabel("Performance difference\n (Δ normalised AUPRC, integrated − baseline)",
              fontsize=12)
ymin = min(0.0, df["delta_norm"].min()) - 0.02
ymax = max(df["delta_norm"].max() + 0.04, 0.20)
ax.set_ylim(ymin, ymax)
for xi, d in zip(xs, df["delta_norm"]):
    ax.text(xi, d + (0.008 if d >= 0 else -0.015), f"{d:+.3f}",
            ha="center", va="bottom" if d >= 0 else "top",
            fontsize=8.5, color="#222")
ax.tick_params(axis="x", length=0)

out_png = EXP_FIG / "loco_imaging_transfers.png"
fig.savefig(out_png, dpi=200, bbox_inches="tight")
fig.savefig(out_png.with_suffix(".pdf"), bbox_inches="tight")
fig.savefig(out_png.with_suffix(".svg"), bbox_inches="tight")
print(f"saved {out_png}")

In [ ]:
# LOCO imaging transfers for ALL domains (Fig S14): per-domain scatter of
# baseline vs integrated normalised AUPRC per held-out cohort. Prevalence per
# (target, fold) = Majority-model AUPRC (as in the normalised LOCO panel, Fig S12a).

maj_prev = (loco_all[loco_all["Model"] == "Majority"]
            .groupby(["Target", "Fold"])["AUPRC"].mean().to_dict())
base_all = (loco_all[loco_all["Model"] == "Baseline"]
            .groupby(["Target", "Fold"])["AUPRC"].mean().to_dict())
stk_all = (stacker_loco.groupby(["Target", "Fold"])["AUPRC"].mean().to_dict())

# domain-specific targets only — the any-domain PSCI scatter stays in Fig S12b
targets_all = [tt for tt in VARIABLE_LABELS
               if tt in {k[0] for k in stk_all}
               and tt != "cognition-PSCI_present"]
rows_ad, wilcox_rows = [], []
for tgt in targets_all:
    fold_to_cohort = loco_fold_cohorts(tgt)
    folds = sorted({k[1] for k in stk_all if k[0] == tgt}
                   & {k[1] for k in base_all if k[0] == tgt}
                   & {k[1] for k in maj_prev if k[0] == tgt})
    for f in folds:
        prev = maj_prev[(tgt, f)]
        if pd.isna(prev) or (1 - prev) < 1e-9:
            continue
        rows_ad.append({
            "target": tgt, "fold": f,
            "cohort": fold_to_cohort[f],
            "prev": prev,
            "baseline_norm": (base_all[(tgt, f)] - prev) / (1 - prev),
            "stacker_norm": (stk_all[(tgt, f)] - prev) / (1 - prev)})
ad = pd.DataFrame(rows_ad)
ad["cohort_display"] = ad["cohort"].map(COHORT_LABELS).fillna(ad["cohort"])
ad.to_csv(EXP / "loco_transfers_all_domains.csv", index=False)

n_t = len(targets_all)
n_cols_ad = 3
n_rows_ad = int(np.ceil(n_t / n_cols_ad))
fig, axes = plt.subplots(n_rows_ad, n_cols_ad,
                         figsize=(5.4 * n_cols_ad, 5.2 * n_rows_ad))
axes = np.atleast_1d(axes).ravel()
for ax, tgt in zip(axes, targets_all):
    sub = ad[ad["target"] == tgt]
    lo = min(sub["baseline_norm"].min(), sub["stacker_norm"].min()) - 0.05
    hi = max(sub["baseline_norm"].max(), sub["stacker_norm"].max()) + 0.05
    ax.plot([lo, hi], [lo, hi], ls=":", color="#888", lw=1.0, zorder=1)
    ax.scatter(sub["baseline_norm"], sub["stacker_norm"], s=70,
               color="#2b6ca3", edgecolor="black", linewidth=0.8, zorder=3)
    texts = [ax.text(r["baseline_norm"], r["stacker_norm"],
                     r["cohort_display"], fontsize=8)
             for _, r in sub.iterrows()]
    adjust_text(texts, ax=ax,
                arrowprops=dict(arrowstyle="-", color="#999", lw=0.5))
    w_ad, p_ad = wilcoxon(sub["stacker_norm"], sub["baseline_norm"],
                          alternative="greater")
    wilcox_rows.append({"target": tgt, "n_cohorts": len(sub),
                        "W": float(w_ad), "p": float(p_ad)})
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_aspect("equal")
    ax.set_title(f"{VARIABLE_LABELS[tgt]}  (n={len(sub)} cohorts, "
                 f"p={p_ad:.3f})", fontsize=12, fontweight="bold")
    ax.set_xlabel("Baseline (normalised AUPRC)", fontsize=10)
    ax.set_ylabel("Integrated (normalised AUPRC)", fontsize=10)
for ax in axes[n_t:]:
    ax.set_visible(False)
wdf = pd.DataFrame(wilcox_rows)
wdf["p_adj_BH"] = bh_fdr(wdf["p"].values)
wdf.to_csv(EXP / "loco_transfers_all_domains_wilcoxon.csv", index=False)
print(wdf.round(4).to_string(index=False))
plt.tight_layout()
out_ad = EXP_FIG / "loco_imaging_transfers_all_domains.png"
fig.savefig(out_ad, dpi=200, bbox_inches="tight")
fig.savefig(out_ad.with_suffix(".pdf"), bbox_inches="tight")
fig.savefig(out_ad.with_suffix(".svg"), bbox_inches="tight")
print(f"saved {out_ad}")

In [ ]:
# Age- and sex-stratified ΔAUPRC strips (Figs S6, S7; mirrors the volume strip)
def strata_delta_strip(strata_dir, order, labels, colors, out_png, legend_ncol=4):
    """Integrated − baseline ΔAUPRC per stratum × target, with NB stars."""
    out_png.parent.mkdir(parents=True, exist_ok=True)
    s_all = pd.read_csv(strata_dir / "all_scores.csv")
    s_stk = pd.read_csv(strata_dir / "stacking_scores.csv")

    nb_rows = []
    for stratum in order:
        df_a = s_all[s_all["Stratum"] == stratum]
        df_s = s_stk[s_stk["Stratum"] == stratum]
        if df_a.empty or df_s.empty:
            continue
        base = df_a[df_a["Modality"] == "base"].copy()
        base["Group"] = base["Model"]
        stk_s = df_s.copy()
        stk_s["Group"] = "Stacking"
        pl = pd.concat([base[["Target", "Group", "Fold", "AUPRC", "AUPRC_train"]],
                        stk_s[["Target", "Group", "Fold", "AUPRC", "AUPRC_train"]]],
                       ignore_index=True)
        nb_s = compute_nb_sig_table(pl, focal_groups=["Stacking"],
                                    reference_groups=["Baseline"], metric="AUPRC")
        nb_s["Stratum"] = stratum
        nb_rows.append(nb_s)
    nb_all = pd.concat(nb_rows, ignore_index=True)
    # FDR family: the strata within each cognitive target
    nb_all["p_adj_BH"] = nb_all.groupby("Target")["p"].transform(lambda s: bh_fdr(s.values))
    nb_all.to_csv(strata_dir / "nb_significance_per_stratum.csv", index=False)
    nb_lookup = {(r["Stratum"], r["Target"]): r["p_adj_BH"]
                 for _, r in nb_all.iterrows()}

    rows = []
    for stratum in order:
        bl = s_all[(s_all["Stratum"] == stratum) & (s_all["Model"] == "Baseline")
                   & (s_all["Modality"] == "base")]
        stk_s = s_stk[s_stk["Stratum"] == stratum]
        for target in bl["Target"].unique():
            b = bl[bl["Target"] == target].set_index("Fold")["AUPRC"]
            s = stk_s[stk_s["Target"] == target].set_index("Fold")["AUPRC"]
            common = b.index.intersection(s.index)
            d = (s.loc[common] - b.loc[common]).dropna()
            if len(d) < 2:
                continue
            mean = float(d.mean())
            # 95% CI with the Nadeau-Bengio corrected variance (as in the test)
            ci = t_dist.ppf(0.975, len(d) - 1) * float(
                np.sqrt((1 / len(d) + 0.1 / 0.9) * d.var(ddof=1)))
            rows.append({"stratum": stratum, "target": target, "delta": mean,
                         "ci_lo": mean - ci, "ci_hi": mean + ci})
    df = pd.DataFrame(rows)

    targets_list = [tt for tt in VARIABLE_LABELS if tt in df["target"].unique()]
    n_targets = len(targets_list)
    x_pos = np.arange(n_targets)
    n_strata = len(order)
    offs = np.linspace(-0.22, 0.22, n_strata) if n_strata > 1 else [0.0]

    sns.set_theme(style="whitegrid", font_scale=1.3,
                  rc={"font.weight": "bold", "axes.labelweight": "bold",
                      "axes.titleweight": "bold"})
    fig, ax = plt.subplots(figsize=(20, 7))
    for o_idx, stratum in enumerate(order):
        sub = df[df["stratum"] == stratum].set_index("target")
        xs = x_pos + offs[o_idx]
        ys = np.array([sub.loc[tt, "delta"] if tt in sub.index else np.nan
                       for tt in targets_list])
        lo = np.array([sub.loc[tt, "ci_lo"] if tt in sub.index else np.nan
                       for tt in targets_list])
        hi = np.array([sub.loc[tt, "ci_hi"] if tt in sub.index else np.nan
                       for tt in targets_list])
        ax.errorbar(xs, ys, yerr=[ys - lo, hi - ys], fmt="o", ms=12,
                    mfc=colors[stratum], mec="black", mew=1.0,
                    ecolor=colors[stratum], elinewidth=2.0,
                    capsize=4, zorder=3, label=labels[stratum])
        for xi, tt, yi, hh in zip(xs, targets_list, ys, hi):
            if np.isnan(yi):
                continue
            p = nb_lookup.get((stratum, tt), np.nan)
            if pd.notna(p) and p < 0.05:
                ax.text(xi, hh + 0.005, "*", ha="center", va="bottom",
                        fontsize=16, fontweight="bold", color="#222")

    ax.axhline(0, color="black", lw=0.6, alpha=0.6)
    # headroom so the significance stars (text, ignored by autoscale) fit
    y_hi = float(np.nanmax(df["ci_hi"].values))
    y_lo = float(np.nanmin(df["ci_lo"].values))
    span = max(y_hi - y_lo, 1e-6)
    ax.set_ylim(min(y_lo, 0) - 0.05 * span, y_hi + 0.18 * span)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([VARIABLE_LABELS[tt] for tt in targets_list],
                       fontsize=13, rotation=0)
    ax.set_ylabel("Performance difference \n(ΔAUPRC, integrated − baseline)")
    ax.grid(True, axis="y", alpha=0.25, ls=":")
    star_proxy = Line2D([0], [0], marker="", linestyle="",
                        label="$*$  > Baseline  ($p_{FDR}<0.05$, NB t-test)")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.20), ncol=legend_ncol,
              fontsize=14, frameon=True, handlelength=1.5, handletextpad=0.6,
              columnspacing=1.5,
              handles=[*ax.get_legend_handles_labels()[0], star_proxy])
    plt.tight_layout()
    fig.savefig(out_png, dpi=200, bbox_inches="tight")
    fig.savefig(out_png.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(out_png.with_suffix(".svg"), bbox_inches="tight")
    print(f"saved {out_png}")
    return nb_all


AGE_STRATA_DIR = PROJECT_ROOT / "outputs" / "predictive_modeling_age_strata"
SEX_STRATA_DIR = PROJECT_ROOT / "outputs" / "predictive_modeling_sex_strata"

if (AGE_STRATA_DIR / "all_scores.csv").exists():
    _age_order = ["young", "middle", "old"]
    strata_delta_strip(
        AGE_STRATA_DIR, _age_order,
        {"young": "Youngest tertile", "middle": "Middle tertile",
         "old": "Oldest tertile"},
        {"young": "#7FB069", "middle": "#E6AA68", "old": "#C75A4A"},
        AGE_STRATA_DIR / "figures" / "age_strata_delta_strip.png")
else:
    print("[skip] age strata outputs missing — run notebook 10 first")

if (SEX_STRATA_DIR / "all_scores.csv").exists():
    _sex_order = sorted(pd.read_csv(SEX_STRATA_DIR / "all_scores.csv")["Stratum"].unique())
    strata_delta_strip(
        SEX_STRATA_DIR, _sex_order,
        # demographics-sex coding: 0 = female, 1 = male (cohort shares match Table 1)
        {s: {"sex_0": "Female", "sex_1": "Male"}.get(s, s) for s in _sex_order},
        {s: c for s, c in zip(_sex_order, ["#5e8eb6", "#c62828", "#888888"])},
        SEX_STRATA_DIR / "figures" / "sex_strata_delta_strip.png", legend_ncol=3)
else:
    print("[skip] sex strata outputs missing — run notebook 10 first")

In [ ]:
# LOCO driver analysis (concise; Fig S12 input): which cohort characteristics
# correlate with per-cohort normalised LOCO performance? Spearman rho of
# normalised AUPRC (integrated) against cohort-level factors (descriptive;
# no significance testing with 13 cohorts).
from scipy.stats import spearmanr

res_drv = pd.read_csv(CACHE)  # per-cohort normalised AUPRC etc.
clin_drv = clinical_df  # loaded earlier in this notebook

factors = {}
for cohort, sub in clin_drv.groupby("demographics-cohort"):
    psci = sub["cognition-PSCI_present"]
    factors[cohort] = {
        "cohort_n": float(psci.notna().sum()),
        "prevalence": float(psci.mean()),
        "age_median": float(pd.to_numeric(sub["demographics-age"], errors="coerce").median()),
        "volume_median": float(pd.to_numeric(
            sub["stroke-total_infarct_volume_ml"], errors="coerce").median()),
        "education_median": float(pd.to_numeric(
            sub["demographics-education_STROKOG"], errors="coerce").median()),
        "nihss_median": float(pd.to_numeric(
            sub["stroke-NIHSS"], errors="coerce").median()),
        "pct_male": float((pd.to_numeric(sub["demographics-sex"],
                                           errors="coerce") == 1).mean()),
        "missing_rate": float(sub[[c for c in sub.columns
                                   if c.startswith(("demographics-", "stroke-",
                                                    "vascularrisk-"))]]
                              .isna().mean().mean()),
    }
fac_df = pd.DataFrame(factors).T
# avoid column collisions with res_drv (which has its own n/prev columns)
fac_df = fac_df.drop(columns=[c for c in fac_df.columns
                              if c in res_drv.columns], errors="ignore")
drv = res_drv.set_index("cohort").join(fac_df, how="inner")
print(f"driver table: {drv.shape[0]} cohorts x {fac_df.shape[1]} factors")

FACTOR_LABEL = {
    "cohort_n": "Cohort size (N)", "prevalence": "PSCI prevalence",
    "age_median": "Median age", "volume_median": "Median infarct volume",
    "education_median": "Median education", "nihss_median": "Median NIHSS",
    "pct_male": "% male", "missing_rate": "Covariate missingness",
}
rows = []
for f in FACTOR_LABEL:
    ok = drv[[f, "norm_AUPRC"]].dropna()
    if len(ok) < 5:
        continue
    rho = spearmanr(ok[f], ok["norm_AUPRC"]).statistic
    rows.append({"factor": f, "rho": float(rho), "n_cohorts": len(ok)})
drivers = pd.DataFrame(rows).sort_values("rho")
drivers.to_csv(EXP / "loco_driver_correlations.csv", index=False)
print(drivers.round(3).to_string(index=False))

sns.set_theme(style="whitegrid", font_scale=1.5,
              rc={"font.weight": "bold", "axes.labelweight": "bold",
                  "axes.titleweight": "bold"})
fig, ax = plt.subplots(figsize=(9, 6))
ys = np.arange(len(drivers))
bar_colors = ["#aa3333" if r > 0 else "#3333aa" for r in drivers["rho"]]
ax.barh(ys, drivers["rho"], color=bar_colors, edgecolor="black",
        linewidth=0.6, alpha=0.88)
for yi, (_, r) in zip(ys, drivers.iterrows()):
    ax.text(r["rho"] + (0.02 if r["rho"] >= 0 else -0.02), yi,
            f'{r["rho"]:+.2f}', va="center",
            ha="left" if r["rho"] >= 0 else "right",
            fontsize=13, fontweight="bold")
ax.axvline(0, color="black", lw=0.8)
ax.set_yticks(ys)
ax.set_yticklabels([FACTOR_LABEL[f] for f in drivers["factor"]], fontsize=14)
ax.set_xlabel("Spearman ρ with cohort-level normalised AUPRC\n"
              "(integrated model, LOCO)", fontsize=14, fontweight="bold")
ax.set_xlim(-1.05, 1.05)
# (panel title added by the S11 composite)
plt.tight_layout()
out_png = EXP_FIG / "loco_drivers_concise.png"
fig.savefig(out_png, dpi=200, bbox_inches="tight")
fig.savefig(out_png.with_suffix(".pdf"), bbox_inches="tight")
fig.savefig(out_png.with_suffix(".svg"), bbox_inches="tight")
print(f"saved {out_png}")

In [ ]:
# Systematic cohort differences (Fig S12 input): z-scored heatmap of the same
# cohort-level factors, cohorts ordered by normalised LOCO AUPRC (integrated).
order_coh = drv.sort_values("norm_AUPRC").index.tolist()
hm_factors = list(FACTOR_LABEL)
hm_raw = drv.loc[order_coh, hm_factors].astype(float)
hm_z = (hm_raw - hm_raw.mean()) / hm_raw.std(ddof=0)

def _fmt(f, v):
    if pd.isna(v):
        return ""
    if f == "cohort_n":
        return f"{v:.0f}"
    if f in ("prevalence", "pct_male", "missing_rate"):
        return f"{100 * v:.0f}%"
    return f"{v:.1f}"

annot = np.array([[_fmt(f, hm_raw.loc[c, f]) for c in order_coh]
                  for f in hm_factors])

sns.set_theme(style="white", font_scale=1.2,
              rc={"font.weight": "bold", "axes.labelweight": "bold"})
fig, ax = plt.subplots(figsize=(11.5, 6))
sns.heatmap(hm_z[hm_factors].T, cmap="RdBu_r", center=0, vmin=-2.5, vmax=2.5,
            annot=annot, fmt="", annot_kws={"fontsize": 9, "fontweight": "bold"},
            linewidths=0.5, linecolor="white",
            cbar_kws={"label": "z-score (per factor)", "shrink": 0.85}, ax=ax)
ax.set_yticklabels([FACTOR_LABEL[f] for f in hm_factors], rotation=0, fontsize=12)
ax.set_xticklabels([COHORT_LABELS.get(c, c) for c in order_coh],
                   rotation=45, ha="right", fontsize=11)
ax.set_xlabel("Cohorts ordered by normalised LOCO AUPRC (integrated, low → high)",
              fontsize=12, fontweight="bold")
# (panel title added by the S6 composite)
plt.tight_layout()
out_png = EXP_FIG / "loco_cohort_differences.png"
fig.savefig(out_png, dpi=200, bbox_inches="tight")
fig.savefig(out_png.with_suffix(".pdf"), bbox_inches="tight")
fig.savefig(out_png.with_suffix(".svg"), bbox_inches="tight")
print(f"saved {out_png}")

In [ ]:
# Absolute performance in the pre-existing lesion subsets (Fig S22): baseline, integrated,
# + WMH (WMH subset) and + all pre-existing lesion types (union subset).
# Baseline and integrated violins from the WMH subset.
sw = pd.read_csv(BR / "acute_vs_wmh" / "stacker_scores.csv")
su = pd.read_csv(BR / "acute_vs_chronic" / "stacker_scores.csv")
bw = pd.read_csv(BR / "acute_vs_wmh" / "baseline_scores.csv")
cols = ["Target", "Fold", "AUPRC"]
L = pd.concat([
    bw.assign(Group="Baseline")[cols + ["Group"]],
    sw[sw["Arm"] == "a_acute_only"].assign(Group="Integrated")[cols + ["Group"]],
    sw[sw["Arm"] == "b_acute_plus_WMH"].assign(Group="+ WMH")[cols + ["Group"]],
    su[su["Arm"] == "b_acute_plus_chronic"].assign(Group="+ union")[cols + ["Group"]],
], ignore_index=True)

GROUPS = ["Baseline", "Integrated", "+ WMH", "+ union"]
PALV = {"Baseline": PALETTE["Baseline"], "Integrated": PALETTE["Stacking"],
        "+ WMH": "#1b9e77", "+ union": "#e7298a"}
LABV = {"Baseline": "Baseline", "Integrated": "Integrated (acute infarct)",
        "+ WMH": "Integrated + WMH", "+ union": "Integrated + all pre-existing lesion types"}
sns.set_theme(style="whitegrid", font_scale=1.3,
              rc={"font.weight": "bold", "axes.labelweight": "bold"})
tg = [t for t in VARIABLE_LABELS if t in L["Target"].unique()]
fig, ax = plt.subplots(figsize=(20, 7))
sns.violinplot(data=L, x="Target", y="AUPRC", hue="Group", order=tg, hue_order=GROUPS,
               palette=PALV, linewidth=0, saturation=0.7, cut=0, inner=None, width=0.85, ax=ax)
gw = 0.85
off = np.linspace(-gw / 2 + gw / 8, gw / 2 - gw / 8, 4)
rng = np.random.RandomState(0)
ax.set_ylim(0.1, 0.95)
for x, t in enumerate(tg):
    for j, gname in enumerate(GROUPS):
        v = L[(L["Target"] == t) & (L["Group"] == gname)]["AUPRC"].values
        cx = x + off[j]
        ax.scatter(rng.normal(cx, 0.012, len(v)), v, s=6, color="gray", alpha=0.35, zorder=2)
        ax.scatter(cx, np.nanmean(v), color="#222", marker="D", s=45, zorder=3)
        ax.text(cx, 0.93, f"{np.nanmean(v):.2f}", ha="center", va="top",
                fontsize=11, fontweight="bold", rotation=90)
for x in range(1, len(tg)):
    ax.axvline(x - 0.5, color="gray", lw=0.3, alpha=0.4)
ax.set_xticks(range(len(tg)))
ax.set_xticklabels([VARIABLE_LABELS[t] for t in tg], fontsize=13)
ax.set_ylabel("Predictive performance (AUPRC)")
ax.set_xlabel("")
h, l = ax.get_legend_handles_labels()
ax.legend(h, [LABV[x] for x in l], bbox_to_anchor=(0.5, -0.16), loc="upper center",
          ncol=2, frameon=True, fontsize=12)
plt.tight_layout()
out_png = BR_FIG / "preexisting_absolute_performance.png"
fig.savefig(out_png, dpi=200, bbox_inches="tight")
fig.savefig(out_png.with_suffix(".pdf"), bbox_inches="tight")
print(f"saved {out_png}")
